# Praktikum Machine Learning: Implementasi TF-IDF
Notebook ini dibuat untuk mencoba dan memahami konsep **TF-IDF (Term Frequency - Inverse Document Frequency)** di Google Colab.

### Apa itu TF-IDF?
TF-IDF adalah metode pembobotan kata yang sering digunakan dalam Information Retrieval dan Text Mining.
1. **Term Frequency (TF)**: Mengukur seberapa sering sebuah kata muncul dalam suatu dokumen.
2. **Inverse Document Frequency (IDF)**: Mengukur seberapa penting kata tersebut di seluruh korpus/koleksi dokumen. Kata yang muncul di hampir semua dokumen (seperti 'dan', 'yang') akan mendapat bobot IDF yang rendah.
$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

## 1. Install & Import Library
Menggunakan `pandas`, `numpy`, `scikit-learn`, serta `PySastrawi` untuk preprocessing teks bahasa Indonesia.

In [ ]:
# Install Sastrawi untuk stopword bahasa Indonesia (jalankan jika belum ada di Colab)
!pip install PySastrawi

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Semua library berhasil di-import!")

## 2. Menyiapkan Dataset Dokumen
Menyiapkan kalimat/dokumen contoh bertema teknologi dan perkuliahan.

In [ ]:
# Kumpulan dokumen contoh (Korpus)
dokumen = [
    "Kecerdasan buatan dan machine learning berkembang sangat pesat di era digital.",
    "Mahasiswa belajar machine learning untuk menyelesaikan tugas akhir dan skripsi.",
    "Python adalah bahasa pemrograman yang populer untuk data science dan machine learning.",
    "Teknologi kecerdasan buatan memudahkan pengolahan data dalam skala besar."
]

# Tampilkan dokumen
df_dokumen = pd.DataFrame({
    'Dokumen_ID': [f'Dokumen {i+1}' for i in range(len(dokumen))],
    'Teks': dokumen
})
df_dokumen

## 3. Text Preprocessing (Pembersihan Teks)
Tahapan pembersihan teks:
1. Case Folding: Mengubah huruf menjadi kecil (lowercase).
2. Punctuation Removal: Menghapus tanda baca dan simbol.
3. Stopword Removal: Menghapus kata umum bahasa Indonesia menggunakan Sastrawi.

In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

def preprocess_teks(teks):
    # 1. Case folding
    teks = teks.lower()
    # 2. Hapus tanda baca dan simbol
    teks = re.sub(r'[^a-z\s]', '', teks)
    # 3. Hapus stopword (kata umum)
    teks = stopword_remover.remove(teks)
    # 4. Rapikan spasi
    teks = ' '.join(teks.split())
    return teks

dokumen_clean = [preprocess_teks(d) for d in dokumen]

df_clean = pd.DataFrame({
    'Dokumen_ID': [f'Dokumen {i+1}' for i in range(len(dokumen))],
    'Teks_Asli': dokumen,
    'Teks_Bersih': dokumen_clean
})
df_clean

## 4. Implementasi TF-IDF (TfidfVectorizer)
Ekstraksi fitur TF-IDF menggunakan class `TfidfVectorizer` dari `scikit-learn`.

In [ ]:
# Inisialisasi TfidfVectorizer
vectorizer = TfidfVectorizer()

# Fit dan transform dokumen yang sudah dibersihkan
tfidf_matrix = vectorizer.fit_transform(dokumen_clean)

# Ambil daftar kata unik (fitur/vocabulary)
feature_names = vectorizer.get_feature_names_out()

# Konversi matriks ke DataFrame pandas
df_tfidf_sklearn = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names,
    index=[f'Dokumen {i+1}' for i in range(len(dokumen))]
)

print(f"Total kata unik (Fitur): {len(feature_names)}")
df_tfidf_sklearn

## 5. Analisis Kata Kunci per Dokumen
Melihat kata dengan bobot TF-IDF tertinggi di setiap dokumen.

In [ ]:
for doc_name, row in df_tfidf_sklearn.iterrows():
    top_words = row[row > 0].sort_values(ascending=False).head(3)
    print(f"\nKata kunci utama di {doc_name}:")
    for word, score in top_words.items():
        print(f"   - {word:<15}: {score:.4f}")

## 6. Penerapan Information Retrieval (Cosine Similarity)
Mencari dokumen yang paling relevan terhadap query pencarian menggunakan Cosine Similarity.

In [ ]:
def cari_dokumen(query, top_n=3):
    # Bersihkan query
    query_clean = preprocess_teks(query)
    
    # Transform query dengan vectorizer
    query_vec = vectorizer.transform([query_clean])
    
    # Hitung cosine similarity
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Buat dataframe ranking
    hasil = pd.DataFrame({
        'Dokumen': [f'Dokumen {i+1}' for i in range(len(dokumen))],
        'Teks': dokumen,
        'Skor Kemiripan': scores
    }).sort_values(by='Skor Kemiripan', ascending=False)
    
    print(f"Hasil pencarian untuk: '{query}'")
    return hasil.head(top_n)

# Uji coba pencarian
cari_dokumen("belajar bahasa python untuk data science")

### Kesimpulan Praktikum:
1. Kata yang sering muncul di semua dokumen memiliki nilai IDF lebih rendah dibanding kata yang spesifik.
2. TfidfVectorizer dari scikit-learn secara default menerapkan normalisasi L2 pada output vektornya.
3. TF-IDF dapat digunakan sebagai representasi fitur teks untuk klasifikasi, clustering, maupun search engine berbasis kemiripan kosinus.